# cDFT + PC-SAFT  vs  TabPFN (with HPO) — map timing & accuracy  (portable)

**Self-contained**: everything needed is in `inputs/` (model + 5 feed CSVs + cDFT times),
so this folder can be copied to any HPC with the `py_a6`/TabPFN environment — it does **not**
need `feos` or the `SENSITIVITY_ANALYSIS` tree.

For each feed it loads the saved **with-HPO TabPFN γ** model and `predict`s the **identical**
(T, P, z) points the cDFT engine evaluated, then compares cDFT map time (recorded) vs ML
`predict` time (live, CUDA-synced), with R²/RMSE on those points.

In [ ]:
import os, json, time
import numpy as np, pandas as pd
n_cpus = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 4))
for v in ("OMP_NUM_THREADS","MKL_NUM_THREADS","OPENBLAS_NUM_THREADS","NUMEXPR_NUM_THREADS"):
    os.environ[v] = str(n_cpus)
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"
import torch, joblib
from sklearn.metrics import r2_score, mean_squared_error
try: torch.set_num_threads(n_cpus)
except Exception: pass
CUDA = torch.cuda.is_available()
print("CUDA:", CUDA, (torch.cuda.get_device_name(0) if CUDA else ""))

In [ ]:
INPUTS = "inputs"
meta      = json.load(open(os.path.join(INPUTS, "cdft_times.json")))
FEATURES  = meta["features"]
DURATIONS = meta["durations"]          # {feed_index(str): cDFT map seconds}
FEEDS     = sorted(int(k) for k in DURATIONS)
print("model:", meta.get("model"), "| target:", meta.get("target"))
print("features:", FEATURES)
print("feeds:", FEEDS)

In [ ]:
t0 = time.perf_counter()
model = joblib.load(os.path.join(INPUTS, "model.joblib"))
print(f"model loaded in {time.perf_counter()-t0:.2f} s -> {type(model).__name__}")

In [ ]:
def predict_timed(X):
    t0 = time.perf_counter()
    yp = model.predict(X)
    if CUDA: torch.cuda.synchronize()
    return yp, time.perf_counter() - t0

rows = []
for i in FEEDS:
    df = pd.read_csv(os.path.join(INPUTS, f"feed_{i}.csv"))
    X, y_true = df[FEATURES], df["gamma"].values
    _ = predict_timed(X.iloc[:16])                  # warm-up
    y_pred, ml_s = predict_timed(X)
    t_cdft, n = float(DURATIONS[str(i)]), len(df)
    rows.append({"feed": i, "points": n,
                 "cDFT_map_time_s": round(t_cdft, 1),
                 "cDFT_per_point_s": round(t_cdft/n, 4),
                 "ML_map_time_s": round(ml_s, 4),
                 "ML_per_point_ms": round(ml_s/n*1e3, 5),
                 "speedup": round(t_cdft/ml_s, 1) if ml_s else np.nan,
                 "R2": round(r2_score(y_true, y_pred), 5),
                 "RMSE": round(np.sqrt(mean_squared_error(y_true, y_pred)), 5)})
    print(f"feed {i}: {n} pts | cDFT {t_cdft:.0f}s | ML {ml_s:.3f}s | "
          f"speedup {t_cdft/ml_s:,.0f}x | R2 {rows[-1]['R2']} | RMSE {rows[-1]['RMSE']}")

res = pd.DataFrame(rows)
res.to_csv("cdft_vs_ml_timing.csv", index=False)
res

In [ ]:
LAB = [("cDFT map time [s]","cDFT_map_time_s"), ("ML map time [s]","ML_map_time_s"),
       ("Speed-up [x]","speedup"), ("ML inference [ms/pt]","ML_per_point_ms"),
       ("R^2","R2"), ("RMSE [mN/m]","RMSE")]
tbl = pd.DataFrame({lab: res.set_index("feed")[key] for lab, key in LAB}).T
tbl.columns = [f"feed {c}" for c in tbl.columns]
display(tbl)
with open("cdft_vs_ml_table.tex", "w") as f:
    f.write(tbl.to_latex(float_format="%.4g",
            caption="cDFT+PC-SAFT vs with-HPO TabPFN: $\\gamma$-map time and accuracy",
            label="tab:cdft_vs_ml"))
print("wrote cdft_vs_ml_timing.csv + cdft_vs_ml_table.tex")